In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
import spotipy
from spotipy.oauth2 import SpotifyClientCredentials

In [ ]:
# ── Spotify API Setup ──
client_id = 'ab30884cbf2b41fda2e26aab4e3a7751'
client_secret = '06885b40fc684369a94080c5b1d87d45'

auth_manager = SpotifyClientCredentials(
    client_id=client_id,
    client_secret=client_secret
)

sp = spotipy.Spotify(auth_manager=auth_manager)

print("Connected to Spotify API")

In [ ]:
# ── Load and clean dataset ──
df = pd.read_csv("dataset.csv")

df["track_name"] = df["track_name"].str.lower()
df["album_name"] = df["album_name"].str.lower()
df["artists"]    = df["artists"].str.lower()
df = df.drop_duplicates(subset=["track_id"])

print(f"Dataset loaded: {len(df)} tracks")

In [ ]:
# ── Audio features used for similarity ──
features = [
    "danceability", "energy", "loudness", "speechiness",
    "acousticness", "instrumentalness", "liveness", "valence", "tempo"
]

---
## 🎧 Enter a Song Name
Search for a song on Spotify, pick the correct match, and get top 10 recommendations.

In [ ]:
# ── Search for a song ──
print('Enter the song name you want recommendations for:')
SONG_NAME   = input('Song name : ').strip()
ARTIST_NAME = input('Artist name (optional, press Enter to skip): ').strip()

query = f'track:{SONG_NAME}'
if ARTIST_NAME:
    query += f' artist:{ARTIST_NAME}'

search_results = sp.search(q=query, type='track', limit=5)
tracks = search_results['tracks']['items']

if not tracks:
    print('Song not found. Try a different name.')
else:
    for i, t in enumerate(tracks):
        artists = ', '.join([a['name'] for a in t['artists']])
        print(f'[{i}] {t["name"]} — {artists}  (id: {t["id"]})')

In [ ]:
# ── Pick the correct result (change PICK_INDEX if needed) ──
PICK_INDEX = 0

chosen_track = tracks[PICK_INDEX]
chosen_track_id = chosen_track["id"]
chosen_track_name = chosen_track["name"]
print(f"✅ Selected: {chosen_track_name} (id: {chosen_track_id})")

In [ ]:
# ── Match the chosen song in our dataset ──
matched = df[df["track_id"] == chosen_track_id]

if matched.empty:
    print("⚠️  Song not found in the local dataset by track_id.")
    print("   Trying by name...")
    matched = df[df["track_name"].str.lower() == chosen_track_name.lower()]

if matched.empty:
    print("❌ Song not in dataset. Try a different song.")
else:
    print(f"✅ Found in dataset: {matched.iloc[0]['track_name']} — {matched.iloc[0]['artists']}")

In [ ]:
# ── Compute similarity & recommend top 10 ──
song_vector = matched.iloc[[0]][features].values  # shape (1, 9)

# Exclude the chosen song itself
other_songs = df[df["track_id"] != matched.iloc[0]["track_id"]].copy()

# Compute cosine similarity
other_features = other_songs[features].values
sim_scores = cosine_similarity(song_vector, other_features)[0]

# Top 10
top_idx = np.argsort(sim_scores)[::-1][:10]
recommendations = other_songs.iloc[top_idx][["track_name", "artists", "album_name"]].reset_index(drop=True)

print(f"\n🎧 Top 10 songs similar to '{chosen_track_name}':\n")
print(recommendations.to_string(index=True))